
Add a structure profile (difficulty, reasoning steps, format, skills) to every RouterBench prompt.
The output `annotations.parquet` must line up row-for-row with the cluster's `prompts.parquet`.


[Needed]
 `data/prompts.parquet` and `data/arrays.npz` to Google Drive in `sta221/data`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/sta221"
DATA_DIR = f"{DRIVE_DIR}/data"
PARTS_DIR = f"{DATA_DIR}/annotation_parts"
os.makedirs(PARTS_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))       # should show prompts.parquet and arrays.npz

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['prompts.parquet', 'arrays.npz', 'annotation_parts']


In [3]:
!pip -q install vllm==0.11.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [9]:
!pip -q install "transformers>=4.56,<5"
import transformers; print(transformers.__version__)   # should print 4.5x / 4.6x, not 5.x

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
5.22.1


In [2]:
import torch, vllm
print("torch", torch.__version__, "| vllm", vllm.__version__)


torch 2.8.0+cu128 | vllm 0.11.0
NVIDIA A100-SXM4-80GB | 85 GB


In [3]:
import hashlib
import pandas as pd

prompts = pd.read_parquet(f"{DATA_DIR}/prompts.parquet")
fingerprint = hashlib.md5("".join(prompts["text"]).encode()).hexdigest()
print(len(prompts), fingerprint)

36481 8bd70fc5726a1baafa781f52cbf24da4


In [4]:
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download("Qwen/Qwen2.5-72B-Instruct-AWQ", local_dir="/content/models/qwen72b-awq")
print("model at", MODEL_DIR)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

model at /content/models/qwen72b-awq



Difficulty is judged for an LLM, not a person. We'll output the following structure feature for each prompt. Which will be fed to the LLM during the RAG phase.

In [5]:
INSTRUCTION = """Classify the task below. Do not solve it.

difficulty: how hard this task is for a typical large language model (not a person).
  easy = most LLMs would answer correctly; medium = mid-size LLMs often fail;
  hard = only the strongest LLMs would answer correctly.
reasoning_steps: number of distinct reasoning or calculation steps needed to reach the answer.
format: multiple_choice, free_text, code, or math_answer.
skills: up to 3 skills the task requires.

Reply with JSON only.

Task:
{task}"""

SCHEMA = {
    "type": "object",
    "properties": {
        "difficulty": {"enum": ["easy", "medium", "hard"]},
        "reasoning_steps": {"type": "integer", "minimum": 0, "maximum": 15},
        "format": {"enum": ["multiple_choice", "free_text", "code", "math_answer"]},
        "skills": {"type": "array", "maxItems": 3, "items": {"enum": [
            "arithmetic", "multi_step_math", "commonsense", "world_knowledge",
            "reading", "coding", "logic", "ethics", "other"]}},
    },
    "required": ["difficulty", "reasoning_steps", "format", "skills"],
}

DEFAULT = {"difficulty": "medium", "reasoning_steps": 1, "format": "free_text", "skills": ["other"]}
MAX_PROMPT_CHARS = 2000
PART_SIZE = 2000

In [6]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from vllm import LLM, SamplingParams

try:
    from vllm.sampling_params import StructuredOutputsParams
    params = SamplingParams(temperature=0, max_tokens=100,
                            structured_outputs=StructuredOutputsParams(json=SCHEMA))
except ImportError:
    from vllm.sampling_params import GuidedDecodingParams
    params = SamplingParams(temperature=0, max_tokens=100,
                            guided_decoding=GuidedDecodingParams(json=SCHEMA))

llm = LLM(MODEL_DIR, max_model_len=2048, max_num_seqs=64,
          gpu_memory_utilization=0.85, enable_prefix_caching=True)

INFO 11-12 22:35:17 [__init__.py:216] Automatically detected platform cuda.
INFO 11-12 22:35:25 [utils.py:233] non-default args: {'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'model': '/content/models/qwen72b-awq'}
INFO 11-12 22:35:25 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-12 22:35:25 [model.py:1510] Using max model len 2048
INFO 11-12 22:35:29 [awq_marlin.py:119] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 11-12 22:35:29 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 11-12 22:35:30 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 11-12 22:38:01 [llm.py:306] Supported_tasks: ['generate']



Each part of 2,000 prompts is saved to Drive. Finished parts are skipped on rerun. Chunking so if the session disconnects we don't lose all work.

In [7]:
import json

for start in range(0, len(prompts), PART_SIZE):
    part_file = f"{PARTS_DIR}/part_{start:06d}.parquet"
    if os.path.exists(part_file):
        continue

    texts = prompts["text"].iloc[start:start + PART_SIZE]
    conversations = [[{"role": "user", "content": INSTRUCTION.format(task=t[:MAX_PROMPT_CHARS])}]
                     for t in texts]

    rows, failures = [], 0
    for out in llm.chat(conversations, params, use_tqdm=False):
        try:
            rows.append(json.loads(out.outputs[0].text))
        except Exception:
            rows.append(DEFAULT)
            failures += 1

    pd.DataFrame(rows).to_parquet(part_file)
    print(f"saved rows {start}-{start + len(rows) - 1} ({failures} failures)", flush=True)

INFO 11-12 22:38:18 [chat_utils.py:560] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
saved rows 0-1999 (0 failures)
saved rows 2000-3999 (0 failures)
saved rows 4000-5999 (0 failures)
saved rows 6000-7999 (0 failures)
saved rows 8000-9999 (0 failures)
saved rows 10000-11999 (0 failures)
saved rows 12000-13999 (0 failures)
saved rows 14000-15999 (0 failures)
saved rows 16000-17999 (0 failures)
saved rows 18000-19999 (0 failures)
saved rows 20000-21999 (0 failures)
saved rows 22000-23999 (0 failures)
saved rows 24000-25999 (0 failures)
saved rows 26000-27999 (0 failures)
saved rows 28000-29999 (0 failures)
saved rows 30000-31999 (0 failures)
saved rows 32000-33999 (0 failures)
saved rows 34000-35999 (0 failures)
saved rows 36000-36480 (0 failures)


In [8]:
import numpy as np
from scipy.stats import spearmanr

part = pd.read_parquet(f"{PARTS_DIR}/part_000000.parquet")
Y = np.load(f"{DATA_DIR}/arrays.npz")["Y"][:len(part)]
n_correct = Y.sum(axis=1)
level = part["difficulty"].map({"easy": 0, "medium": 1, "hard": 2})

print(part["difficulty"].value_counts())
print("difficulty vs # correct:", round(spearmanr(level, n_correct).correlation, 3))
print("steps vs # correct:     ", round(spearmanr(part["reasoning_steps"], n_correct).correlation, 3))
arc = (prompts["eval_name"][:len(part)] == "arc-challenge").values
print("within ARC, difficulty vs # correct:", round(spearmanr(level[arc], n_correct[arc]).correlation, 3))
for n in [0, 500, 1500]:
    print("\n", prompts["text"][n][:150], "\n  ->", part.iloc[n].to_dict())

difficulty
easy      1476
medium     522
hard         2
Name: count, dtype: int64
difficulty vs # correct: -0.063
steps vs # correct:      -0.227
within ARC, difficulty vs # correct: -0.144

 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 古字刚分开，豆字挤进来，你先别生气，看后笑开怀。 
  -> {'difficulty': 'medium', 'reasoning_steps': 3, 'format': 'free_text', 'skills': array(['logic', 'coding', 'commonsense'], dtype=object)}

 An exothermic reaction could best be demonstrated with a
A) car engine.
B) glass bottle.
C) frying pan.
D) refrigerator.
Print only a single choice  f 
  -> {'difficulty': 'easy', 'reasoning_steps': 1, 'format': 'multiple_choice', 'skills': array(['commonsense', 'commonsense'], dtype=object)}

 Experiments show that mice can learn to press a lever in order to acquire food. This learning ability can help mice survive in the wild by allowing th 
  -> {'difficulty': 'easy', 'reasoning_steps': 1, 'format': 'multiple_choice', 'skills': array(['commonsense', 're

In [9]:
part_files = sorted(f for f in os.listdir(PARTS_DIR) if f.endswith(".parquet"))
annotations = pd.concat([pd.read_parquet(f"{PARTS_DIR}/{f}") for f in part_files], ignore_index=True)

if len(annotations) == len(prompts):
    annotations.to_parquet(f"{DATA_DIR}/annotations.parquet")
    print("saved", len(annotations), "annotations")
    print(annotations["difficulty"].value_counts())
    print("annotator: Qwen2.5-72B-Instruct-AWQ | vllm", vllm.__version__)
else:
    print(f"{len(annotations)} of {len(prompts)} done; rerun Cell 8 to continue")

saved 36481 annotations
difficulty
easy      27453
medium     8670
hard        358
Name: count, dtype: int64
annotator: Qwen2.5-72B-Instruct-AWQ | vllm 0.11.0
